In [ ]:
import os
from matplotlib import animation
from IPython.display import HTML

# Load processed GoFlow files for the most recent 3 days based on filename start times
data_dir = '/home/mduplessis/share/EUMETSAT/processed_goflow_inputs/5E-25E_44S-33S'
all_files = sorted(glob.glob(f'{data_dir}/*.nc'))

if len(all_files) == 0:
    raise FileNotFoundError(f'No NetCDF files found in {data_dir}')

file_starts = pd.to_datetime(
    [os.path.basename(p).split('-')[0] for p in all_files],
    format='%Y%m%d%H%M',
    errors='coerce',
)
valid = ~file_starts.isna()
all_files = [p for p, keep in zip(all_files, valid) if keep]
file_starts = file_starts[valid]

end_dt = file_starts.max()
start_dt = end_dt - pd.Timedelta(days=3)
sel_files = [p for p, t in zip(all_files, file_starts) if t >= start_dt]

if len(sel_files) == 0:
    raise ValueError('No files found for the last 3 days.')

ds3 = xr.open_mfdataset(sel_files, combine='nested', concat_dim='time', parallel=True)
ds3 = ds3.sortby('time')
ds3 = ds3.isel(time=~ds3.indexes['time'].duplicated())

# Ensure BT_masked exists
if 'BT_masked' not in ds3:
    bt_c = ds3['BT'] - 273.15 if float(ds3['BT'].mean()) > 100 else ds3['BT']
    ds3['BT_masked'] = bt_c.where(ds3['mask'] == 1)

# Compute rolling 12-hour average in sample space using median timestep
dt_hours = pd.Series(pd.to_datetime(ds3.time.values)).diff().dt.total_seconds().dropna() / 3600.0
if len(dt_hours) == 0:
    raise ValueError('Not enough time points to compute rolling averages.')
step_hours = float(dt_hours.median())
window_n = max(1, int(round(12.0 / step_hours)))

# NaN-aware rolling mean: only require a fraction of valid samples in each 12h window
min_valid = max(1, int(np.ceil(window_n * 0.25)))
bt_roll = (
    ds3['BT_masked']
    .rolling(time=window_n, min_periods=min_valid)
    .mean(skipna=True)
    .dropna('time', how='all')
)
if bt_roll.sizes.get('time', 0) == 0:
    raise ValueError('Rolling 12-hour average produced no frames.')

# Fill remaining gaps with per-pixel 3-day nanmean background to reduce speckled NaN holes
bt_bg = bt_roll.mean(dim='time', skipna=True)
bt_roll_filled = bt_roll.fillna(bt_bg)

# Subsample frames if desired to keep notebook animation responsive
frame_stride = 3
bt_frames = bt_roll_filled.isel(time=slice(None, None, frame_stride))

fig_anim, ax_anim = plt.subplots(
    figsize=(16, 8),
    subplot_kw={'projection': ccrs.PlateCarree()},
)

vmin = float(bt_frames.quantile(0.05, skipna=True).compute())
vmax = float(bt_frames.quantile(0.95, skipna=True).compute())

mesh = ax_anim.pcolormesh(
    ds3['lon'].values,
    ds3['lat'].values,
    bt_frames.isel(time=0).values,
    transform=ccrs.PlateCarree(),
    cmap=cmo.thermal,
    vmin=vmin,
    vmax=vmax,
    shading='auto',
)
cbar = fig_anim.colorbar(mesh, ax=ax_anim, pad=0.02, aspect=30)
cbar.set_label('BT_masked rolling 12h mean [C]')

ax_anim.set_extent([9, 20, -40, -33], crs=ccrs.PlateCarree())
ax_anim.coastlines(resolution='10m', linewidth=0.8, color='black')
grid = ax_anim.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.6,
    color='gray',
    alpha=0.6,
    linestyle='--',
)
grid.top_labels = False
grid.right_labels = False
ax_anim.scatter(15.4270, -34.5001, color='black', s=100, marker='*', transform=ccrs.PlateCarree(), zorder=14)

def draw_eddy_lines_on_ax(frame_date_str, ax):
    lines = []
    cyclones = eddy_data.get(frame_date_str, {}).get('cyclones', [])
    anticyclones = eddy_data.get(frame_date_str, {}).get('anticyclones', [])
    for lon_poly, lat_poly in cyclones:
        line, = ax.plot(
            lon_poly,
            lat_poly,
            color='tab:blue',
            linewidth=1.5,
            transform=ccrs.PlateCarree(),
            zorder=13,
        )
        lines.append(line)
    for lon_poly, lat_poly in anticyclones:
        line, = ax.plot(
            lon_poly,
            lat_poly,
            color='tab:red',
            linewidth=1.5,
            transform=ccrs.PlateCarree(),
            zorder=13,
        )
        lines.append(line)
    return lines

eddy_artists = []

def update(frame_idx):
    global eddy_artists
    for ln in eddy_artists:
        ln.remove()
    eddy_artists = []

    arr = bt_frames.isel(time=frame_idx).values
    mesh.set_array(arr.ravel())

    t0 = pd.to_datetime(bt_frames.time.values[frame_idx])
    t_start = t0 - pd.Timedelta(hours=12)
    date_key = t0.strftime('%Y-%m-%d')
    eddy_artists = draw_eddy_lines_on_ax(date_key, ax_anim)

    ax_anim.set_title(
        f'BT_masked rolling 12h mean: {t_start:%Y-%m-%d %H:%M} to {t0:%Y-%m-%d %H:%M}',
        pad=12,
    )

    return [mesh, *eddy_artists]

anim = animation.FuncAnimation(
    fig_anim,
    update,
    frames=bt_frames.sizes['time'],
    interval=250,
    blit=False,
    repeat=True,
)

HTML(anim.to_jshtml())